# H-008 · Does Gross Profitability Predict Forward Returns?

Factor test for **H-008** (equities): whether high TTM gross profitability (`GP / Assets`) predicts higher forward returns (positive IC) at Alphalens `periods=(1, 5, 21)` (primary narrative **5d**).

- **Idea** — Novy-Marx (2013) gross profitability: TTM GrossProfit (or Revenue − COGS) divided by Total Assets, held PIT from SEC `filed` via `merge_asof` backward.
- **Claim** — High `gp_asset` predicts higher next-week / next-month returns; quality complements value.
- **Why it might work** — GP is upstream of SG&A, interest, and taxes — cleaner than net income. Quality and value are negatively correlated in the cross-section.
- **Data** — Load `s1_factor_panel_train.parquet` (research IS; do not re-split). Merge cached / fetched SEC GP fields on `["date", "ticker"]`. Do **not** use `s1_factor_panel_full.parquet` for keep/kill.

**Baseline (deferred):** This notebook screens **GP only**. Incremental comparison vs H-005 `book_yield` / `earnings_yield` / `log_mcap` (and nested IC / GBM) is a **follow-up** — use the H-005 notebook / its SV cache later. Do not treat a lone GP IC as a keep vs value.

**No floor / no winsorize in the feature store:** `add_gross_profitability_factors` does **not** floor assets (missing or `<= 0` → NaN `gp_asset`) and does **not** winsorize. If you need winsorization, apply it in §2 — not inside the factor API.

**SEC cache:** only the un-edited fetcher output is cached (`s1_h008_gp_sec.parquet`: `date`, `ticker`, `gross_profit_ttm`, `assets`, `gp_asset`). The OHLCV+feature panel is **not** cached.

| Knob | Values |
|------|--------|
| Construction | TTM only (no quarterly mode in v1) |
| Tags | Prefer `GrossProfit`; else Revenue − COGS; Assets → `Assets` else `AssetsCurrent` |
| `normalize` | `True` (fixed for this notebook) → CS pct-rank by date |
| Column | `gross_profitability` |
| Alphalens `periods` | `(1, 5, 21)` (primary narrative **5d**) |
| Tear PDF | `tearsheets/H-008_gross_profitability.pdf` |

Use `data.processing.feature_store.add_gross_profitability_factors` (fetches SEC GP unless `gross_profitability_data_exists=True`).

Evaluation uses the S1 **trade-date** panel: Alphalens pivots `open` (no `shift(-1)`); labels are open-to-open. Prior close-to-close ICs are not comparable.


## 0. Imports & Config

Resolve the repo root; configure `NORMALIZE=True`, Alphalens `PERIODS`, and tearsheet paths. The train parquet is already research IS; do not calculate another cutoff here.


In [1]:
import os
import sys

import alphalens as al
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import pandas as pd

from data.processing.feature_store import add_gross_profitability_factors

# Jupyter cwd is often this notebook's folder, not the repo root; walk up until we find 01_data/ingestion.
ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

TRAIN_PANEL_PATH = os.path.join(
    ROOT, "01_data", "data_files", "s1_equities", "s1_factor_panel_train.parquet"
)
# Un-edited SEC fetcher output only (not the OHLCV+feature panel).
TEARSHEET_DIR = os.path.join(
    ROOT, "02_research", "notebooks", "factor_tests", "tearsheets"
)

FORCE_REBUILD = False
NORMALIZE = True
FACTOR_COL = "gross_profitability"

PERIODS = (1, 5, 21)  # S1 default; primary narrative = 5d
QUANTILES = 5
MAX_LOSS = 0.35

print(f"ROOT={ROOT}")
print(f"FORCE_REBUILD={FORCE_REBUILD}  NORMALIZE={NORMALIZE}  FACTOR_COL={FACTOR_COL}")


ROOT=c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio
FORCE_REBUILD=False  NORMALIZE=True  FACTOR_COL=gross_profitability
GP_SEC_CACHE_PATH=c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\01_data\data_files\s1_equities\s1_h008_gp_sec.parquet


## 1. Data Loading

1. Load `s1_factor_panel_train.parquet`.
2. SEC gross profitability is attached inside
   `add_gross_profitability_factors(..., gross_profitability_data_exists=False)`
   (underlying fetcher cache still applies under `01_data/cache`).
3. Coverage of `gp_asset` is printed after the store call in §3.


In [2]:
panel = pd.read_parquet(TRAIN_PANEL_PATH)
required = {"date", "ticker", "open", "close", "feature_date"}
missing = required - set(panel.columns)
if missing:
    raise ValueError(f"train panel missing columns: {sorted(missing)}")

panel = panel.copy()
panel["date"] = pd.to_datetime(panel["date"])
panel["feature_date"] = pd.to_datetime(panel["feature_date"])
panel["ticker"] = panel["ticker"].astype(str).str.strip().str.upper()
tickers = sorted(panel["ticker"].unique().tolist())
start = panel["date"].min().date()
end = panel["date"].max().date()
print(
    f"train panel: rows={len(panel):,}  tickers={len(tickers)}  "
    f"dates={panel['date'].nunique():,}  [{start} -> {end}]"
)
print(
    "SEC gross profitability attaches inside "
    "add_gross_profitability_factors(gross_profitability_data_exists=False)."
)
panel.head()


train panel: rows=289,381  tickers=100  dates=2,915  [2010-01-05 -> 2021-08-03]
SEC CACHE MISS: fetching Company Facts for 100 tickers [2010-01-05 -> 2021-08-03]


No SEC CIK for ticker AET; leaving GP fundamentals NaN
No SEC CIK for ticker ESRX; leaving GP fundamentals NaN
No SEC CIK for ticker TWX; leaving GP fundamentals NaN


Wrote c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\01_data\data_files\s1_equities\s1_h008_gp_sec.parquet
rows=289,281  gp_asset non-null=32.4%
merge: gp_asset non-null=32.4%  assets=92.9%


,date,ticker,open,high,low,close,volume,feature_date,fwd_ret_1,fwd_ret_5,fwd_ret_21,gross_profit_ttm,assets,gp_asset
0,2010-01-05,AAPL,6.424143,6.421146,6.357683,6.406478,493729600.0,2010-01-04,-0.001025,-0.025210,-0.083271,NaN,NaN,NaN
1,2010-01-06,AAPL,6.417558,6.453779,6.383729,6.417557,601904800.0,2010-01-05,-0.012268,-0.030367,-0.101456,NaN,5.385100e+10,NaN
2,2010-01-07,AAPL,6.338825,6.443003,6.308892,6.315478,552160000.0,2010-01-06,-0.006848,-0.007745,-0.075844,NaN,5.385100e+10,NaN
3,2010-01-08,AAPL,6.295419,6.346309,6.258000,6.303801,477131200.0,2010-01-07,0.011888,0.002996,-0.066001,NaN,5.385100e+10,NaN
4,2010-01-11,AAPL,6.370258,6.346310,6.258300,6.345711,447610800.0,2010-01-08,-0.016965,-0.021005,-0.079464,NaN,5.385100e+10,NaN


## 2. Data Cleaning & Engineering

No floor / no winsorize (match store). Daily `gp_asset` is PIT-held via the
fetcher `merge_asof` inside the store attach — do **not** extra-ffill here.

Coverage of `gp_asset` is reported after §3 attaches the SEC columns.


In [3]:
print(
    "skip §2 coverage — gp_asset is attached in §3 via "
    "add_gross_profitability_factors"
)


rows=289,381  gp_asset coverage=32.4%
by-date coverage: min=0.0%  median=33.0%  max=47.4%
tickers with any gp_asset: 71 / 100


## 3. Modeling / Signal Construction

Call `add_gross_profitability_factors(panel, feature_subset=['gross_profitability'], normalize=NORMALIZE, gross_profitability_data_exists=...)` → column `gross_profitability` (CS pct-rank of `gp_asset` within date when `NORMALIZE=True`). Do not reimplement the ratio inline.


In [4]:
panel = add_gross_profitability_factors(
    panel,
    feature_subset=["gross_profitability"],
    normalize=NORMALIZE,
    gross_profitability_data_exists=False,
)
if FACTOR_COL not in panel.columns:
    raise ValueError(f"expected {FACTOR_COL!r} after store call")
n_ok = panel[FACTOR_COL].notna().sum()
print(f"{FACTOR_COL}: non-null={n_ok:,} ({panel[FACTOR_COL].notna().mean():.1%})")
panel[["date", "ticker", "gp_asset", FACTOR_COL]].dropna(subset=[FACTOR_COL]).head()

# Raw SEC coverage after in-store fetch/merge
if "gp_asset" in panel.columns:
    cov = panel["gp_asset"].notna().mean()
    print(f"gp_asset coverage={cov:.1%}")


gross_profitability: non-null=93,855 (32.4%)


,date,ticker,gp_asset,gross_profitability
14,2010-01-26,AAPL,0.174943,0.433333
15,2010-01-27,AAPL,0.174943,0.433333
16,2010-01-28,AAPL,0.174943,0.451613
17,2010-01-29,AAPL,0.174943,0.451613
18,2010-02-01,AAPL,0.174943,0.466667


## 4. Evaluation

Alphalens IC + quintile spreads on research IS only at `periods=(1, 5, 21)` (primary narrative **5d**) for the single column `gross_profitability`.

**Baseline (deferred):** do not interpret keep/kill vs H-005 value/size from this notebook alone.


### 4.1 IC / spread summary

Primary ranking metric: **mean IC at 5d** (`ic_5d`). One row for `gross_profitability`.


In [5]:
def to_alphalens_prices(panel: pd.DataFrame) -> pd.DataFrame:
    """Wide open matrix for Alphalens (trade-date panel; entry at open)."""
    prices = panel.pivot(index="date", columns="ticker", values="open")
    prices.index = pd.to_datetime(prices.index)
    return prices.sort_index()


def to_alphalens_factor(panel: pd.DataFrame, col: str) -> pd.Series:
    """MultiIndex (date, ticker) factor series for Alphalens."""
    factor = panel.set_index(["date", "ticker"])[col].dropna()
    factor.index = factor.index.set_levels(
        pd.to_datetime(factor.index.levels[0]), level=0
    )
    return factor.sort_index()


def _period_label(period_index: pd.Index, period: int, position: int):
    """Match Alphalens period label ('1D', '5D', …) or fall back by position."""
    for c in (f"{period}D", f"{period}d", period, str(period)):
        if c in period_index:
            return c
    return period_index[position]


def factor_screen_metrics(
    factor: pd.Series,
    prices: pd.DataFrame,
    *,
    periods: tuple[int, ...] = PERIODS,
    quantiles: int = QUANTILES,
    max_loss: float = MAX_LOSS,
) -> dict:
    """Mean IC and Q5−Q1 mean return spread for each forward period."""
    factor_data = al.utils.get_clean_factor_and_forward_returns(
        factor=factor,
        prices=prices,
        quantiles=quantiles,
        periods=periods,
        max_loss=max_loss,
    )
    mean_ic = al.performance.mean_information_coefficient(factor_data)
    mean_ret, _ = al.performance.mean_return_by_quantile(factor_data, demeaned=True)

    row = {}
    for i, p in enumerate(periods):
        ic_key = _period_label(mean_ic.index, p, i)
        ret_key = _period_label(mean_ret.columns, p, i)
        row[f"ic_{p}d"] = float(mean_ic.loc[ic_key])
        q_hi, q_lo = mean_ret.index.max(), mean_ret.index.min()
        row[f"spread_{p}d"] = float(
            mean_ret.loc[q_hi, ret_key] - mean_ret.loc[q_lo, ret_key]
        )
    return row


def run_full_tear(
    panel: pd.DataFrame,
    factor_col: str,
    prices: pd.DataFrame,
    *,
    periods: tuple[int, ...] = PERIODS,
    quantiles: int = QUANTILES,
    max_loss: float = MAX_LOSS,
    tearsheet_dir: str = TEARSHEET_DIR,
):
    """Build factor_data, run Alphalens full tear, display + save multi-page PDF.

    Alphalens calls plt.show() after each plot, which clears figures under Agg.
    Temporarily replace plt.show so each figure is written into the PDF before close.
    """
    if factor_col not in panel.columns:
        raise ValueError(f"{factor_col!r} not in panel columns")
    plt.close("all")
    factor_data = al.utils.get_clean_factor_and_forward_returns(
        factor=to_alphalens_factor(panel, factor_col),
        prices=prices,
        quantiles=quantiles,
        periods=periods,
        max_loss=max_loss,
    )

    os.makedirs(tearsheet_dir, exist_ok=True)
    out_path = os.path.join(tearsheet_dir, f"H-008_{factor_col}.pdf")
    pdf = PdfPages(out_path)
    n_pages = 0
    _original_show = plt.show

    def _show_and_savefig(*args, **kwargs):
        nonlocal n_pages
        for num in list(plt.get_fignums()):
            fig = plt.figure(num)
            if fig.axes:
                pdf.savefig(fig, bbox_inches="tight")
                n_pages += 1
        plt.close("all")

    plt.show = _show_and_savefig
    try:
        al.tears.create_full_tear_sheet(factor_data, long_short=True)
        _show_and_savefig()
    finally:
        plt.show = _original_show
        pdf.close()
        plt.close("all")

    print(f"Wrote {out_path} ({n_pages} pages)")
    return factor_data


In [7]:
prices = to_alphalens_prices(panel)
metrics = factor_screen_metrics(to_alphalens_factor(panel, FACTOR_COL), prices)
summary = pd.DataFrame([{"factor": FACTOR_COL, "normalize": NORMALIZE, **metrics}])

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.4f}".format)
print("Primary metric: ic_5d (deferred baseline = H-005 value/size)")
summary


c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any n

Primary metric: ic_5d (deferred baseline = H-005 value/size)


,factor,normalize,ic_1d,spread_1d,ic_5d,spread_5d,ic_21d,spread_21d
0,gross_profitability,True,0.0112,0.0001,0.0142,0.0003,0.0284,0.0005


### 4.2 Full tear sheet

Runs on `FACTOR_COL` (`gross_profitability`). The tear is displayed in-notebook **and** saved as a multi-page PDF under `02_research/notebooks/factor_tests/tearsheets/` named `H-008_gross_profitability.pdf`. Re-running overwrites the same path.

Do not treat this tear as a keep vs H-005 until the deferred baseline pass.


In [8]:
print(f"Tear sheet factor: {FACTOR_COL}")
tear_data = run_full_tear(panel, FACTOR_COL, prices)


Tear sheet factor: gross_profitability


c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any n

Quantiles Statistics


,min,max,mean,std,count,count %
factor_quantile,,,,,,
1,0.0217,0.2857,0.1218,0.0611,19697,21.1841
2,0.2222,0.4545,0.3241,0.0562,17988,19.3461
3,0.4186,0.6364,0.5181,0.0564,18098,19.4644
4,0.6136,0.8182,0.7122,0.0559,17988,19.3461
5,0.8095,1.0000,0.9121,0.0593,19209,20.6593


Returns Analysis


...

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\performance.py:118: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  grouper.append(pd.Grouper(freq=by_time))
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:928: UserWarning: Skipping return periods that aren't exact multiples of days.
  warnings.warn(


Turnover Analysis


,1D,5D,21D
Quantile 1 Mean Turnover,0.0180,0.0750,0.2480
Quantile 2 Mean Turnover,0.0330,0.1250,0.3650
Quantile 3 Mean Turnover,0.0360,0.1310,0.3760
Quantile 4 Mean Turnover,0.0320,0.1200,0.3410
Quantile 5 Mean Turnover,0.0170,0.0690,0.2220


,1D,5D,21D
Mean Factor Rank Autocorrelation,0.9990,0.9970,0.9870


Wrote c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\02_research\notebooks\factor_tests\tearsheets\H-008_gross_profitability.pdf (3 pages)


## 5. Wrap-up / next steps

Fill after running (§4.1 / tear):

- **`ic_5d` / spreads:** …
- **Coverage:** was `gp_asset` dense enough for a fair CS rank?
- **Keep / kill (H-008 alone):** …
- **Deferred baseline:** side-by-side IC vs H-005 `book_yield` / `earnings_yield` / `log_mcap` (H-005 notebook / SV cache); then nested IC / GBM gain per hypothesis log.
- **SEC data:** rely on fetcher disk cache; pass `gross_profitability_data_exists=True` on repeat store calls in the same session.
